# data import , libraries import,  adjust display options


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import re

#### **Data Imports**
We import `pandas` and `numpy` to handle data manipulation and mathematical analysis.

In [ ]:
pd.set_option('display.max_columns',None)
pd.set_option('display.width',None)
pd.options.display.float_format = '{:,.2f}'.format

#### **Setting Display Formats**
We adjust the default float formatting to display commas for thousands and two decimal places (e.g., `1,000,000.00`), which is ideal when looking at monetary data like car prices.

In [ ]:
cdf=pd.read_csv('/content/drive/MyDrive/DEPI datasets/used_cars.csv')
hdf=pd.read_csv('/content/drive/MyDrive/DEPI datasets/price_history.csv')

In [ ]:
uc=cdf.copy()
ph=hdf.copy()

#### **Why are we keeping copies of the DataFrames?**
Creating copies like `uc` and `ph` allows us to clean and modify the data safely without losing or corrupting our original raw DataFrames (`cdf` and `hdf`).

# Data Exploratory

#### **used cars exploaring**

In [ ]:
uc.shape

(25223, 25)

In [ ]:
uc.columns

Index(['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage', 'Color',
       'Fuel Type', 'Body Shape', 'Transmission', 'Engine', 'Engine Capacity',
       'Cylinder Count', 'Turbo Count', 'HP', 'Drivetrain', 'car_age',
       'miles_per_year', 'Brand_Tier', 'Image Paths', 'Condition',
       'Condition_Score', 'model_average_price', 'model_min_price',
       'model_max_price'],
      dtype='object')

In [ ]:
uc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25223 entries, 0 to 25222
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Brand                25223 non-null  object 
 1   Model                25223 non-null  object 
 2   Trim                 25223 non-null  object 
 3   Year                 25223 non-null  int64  
 4   Price                25223 non-null  int64  
 5   Mileage              25223 non-null  int64  
 6   Color                25223 non-null  object 
 7   Fuel Type            25223 non-null  object 
 8   Body Shape           25223 non-null  object 
 9   Transmission         25223 non-null  object 
 10  Engine               25223 non-null  object 
 11  Engine Capacity      25223 non-null  int64  
 12  Cylinder Count       25223 non-null  int64  
 13  Turbo Count          25223 non-null  int64  
 14  HP                   25223 non-null  int64  
 15  Drivetrain           25223 non-null 

In [ ]:
uc.isnull().sum()

,0
Brand,0
Model,0
Trim,0
Year,0
Price,0
Mileage,0
Color,0
Fuel Type,0
Body Shape,0
Transmission,0


In [ ]:
uc['Year'].value_counts()


,count
Year,
2021,2077
2022,1987
2020,1653
2019,1583
2024,1440
2016,1396
2015,1356
2023,1199
2017,1163


# used_cars Data Cleaning

## Text Normalization

In [ ]:
text_cols = ['Brand', 'Model', 'Trim', 'Color', 'Fuel Type', 'Body Shape', 'Transmission', 'Engine', 'Drivetrain']

for col in text_cols:
    if col in uc.columns:
        # Strip leading/trailing whitespaces and convert to Title Case consistently
        uc[col] = uc[col].astype(str).str.strip().str.title()

print("Text columns successfully standardized to Title Case and stripped of whitespace!")
uc[text_cols].head()

Text columns successfully standardized to Title Case and stripped of whitespace!


,Brand,Model,Trim,Color,Fuel Type,Body Shape,Transmission,Engine,Drivetrain
0,Alfa Romeo,Giulia,Base,Black,Gas,Sedan,Automatic,2.0 L Turbocharged I4,Rwd
1,Alfa Romeo,Giulia,Super,Grey,Gas,Sedan,Automatic,2.0 L Turbocharged I4,Rwd
2,Alfa Romeo,Giulia,Base,Black,Gas,Sedan,Automatic,2.0 L Turbocharged I4,Rwd
3,Alfa Romeo,Giulia,Base,Silver,Gas,Sedan,Automatic,2.0 L Turbocharged I4,Rwd
4,Alfa Romeo,Giulietta,Base,Black,Gas,Hatchback,Automatic,1.4 L Multiair Turbo I4,Fwd


In [ ]:
# edit ruined cells
brand_fix = {'Bmw':'BMW','Byd':'BYD','Mg':'MG','Gmc':'GMC','Jac':'JAC','Jmc':'JMC',
             'Gac':'GAC','Baic':'BAIC','Dfsk':'DFSK','Ds':'DS','Exeed':'EXEED',
             'Kgm':'KGM','Vgv':'VGV','Xpeng':'XPeng','Ssangyong':'SsangYong'}
uc['Brand'] = uc['Brand'].replace(brand_fix)
uc['Body Shape'] = uc['Body Shape'].replace({'Suv':'SUV','Mpv':'MPV'})
uc['Drivetrain'] = uc['Drivetrain'].replace({'Fwd':'FWD','Rwd':'RWD','Awd':'AWD'})

In [ ]:
# Rename the column 'miles_per_year' to 'km_per_year'
if 'miles_per_year' in uc.columns:
    uc = uc.rename(columns={'miles_per_year': 'km_per_year'})
    print("Column successfully renamed!")
else:
    print("Column 'miles_per_year' not found or already renamed.")


Column successfully renamed!


## Duplicates

In [ ]:
uc.duplicated().sum()
# no direct duplicates

np.int64(0)

In [ ]:
duplicates_filter=['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage', 'Color',
       'Fuel Type', 'Body Shape', 'Transmission', 'Engine', 'Engine Capacity',
       'Cylinder Count', 'Turbo Count', 'HP', 'Drivetrain', 'car_age',
       'km_per_year', 'Brand_Tier']

In [ ]:
uc.duplicated(subset=duplicates_filter).sum()

np.int64(1688)

In [ ]:
# dublicates flag
uc['is_duplicated']=uc.duplicated(subset=duplicates_filter,keep='first')

In [ ]:
uc['is_duplicated'].sum()

np.int64(1688)

#### **Investigating Duplicate Listings**
We define a list of columns `duplicates_filter` to identify identical physical cars (same brand, year, price, mileage, transmission, etc.). Standard `.duplicated()` didn't catch them initially because some listings have different image paths, but they are practically duplicate entries.

In [ ]:
# new data frame to visualize the dublicates
dups=uc[uc.duplicated(subset=duplicates_filter,keep=False)].sort_values(duplicates_filter)
cols=['Brand','Model','Year','Price','Mileage','Condition','Condition_Score','Image Paths']
dups[cols].head(30)

,Brand,Model,Year,Price,Mileage,Condition,Condition_Score,Image Paths
16,Alfa Romeo,Giulia,2021,1480000,125000,Very Good,6.30,Used_Images\Alfa_Romeo_Giulia_2021_36056\img_1...
25,Alfa Romeo,Giulia,2021,1480000,125000,Very Good,6.70,Used_Images\Alfa_Romeo_Giulia_2021_288\img_1.j...
6,Alfa Romeo,Mito,2017,800000,130000,Very Good,7.90,Used_Images\Alfa_Romeo_Mito_2017\car_img_1.jpg...
32,Alfa Romeo,Mito,2017,800000,130000,Very Good,7.80,Used_Images\Alfa_Romeo_Mito_2017_27876\img_1.j...
22,Alfa Romeo,Tonale,2024,2700000,10000,Very Good,6.80,Used_Images\Alfa_Romeo_Tonale_2024_42741\img_1...
29,Alfa Romeo,Tonale,2024,2700000,10000,Excellent,8.00,Used_Images\Alfa_Romeo_Tonale_2024_26650\img_1...
21,Alfa Romeo,Tonale,2025,2600000,1350,Excellent,8.30,Used_Images\Alfa_Romeo_Tonale_2025_10569\img_1...
33,Alfa Romeo,Tonale,2025,2600000,1350,Very Good,7.90,Used_Images\Alfa_Romeo_Tonale_2025_21666\img_1...
67,Audi,A3,2013,650000,210000,Good,5.20,Used_Images\Audi_A3_2013\car_img_1.jpg; Used_I...
187,Audi,A3,2013,650000,210000,Good,5.20,Used_Images\Audi_A3_2013_83387\img_1.jpeg; Use...


In [ ]:
# check drop effect
# calculate the difference between every brand's median befor and after droping duplicates
before=uc.groupby('Brand').Price.median()
after=uc[~ uc.is_duplicated].groupby('Brand').Price.median()
after

,Price
Brand,
Alfa Romeo,"1,825,000.00"
Aston Martin,"13,000,000.00"
Audi,"2,150,000.00"
Avatr,"3,075,000.00"
BAIC,"1,050,000.00"
...,...
Wuling,"680,000.00"
XPeng,"2,400,000.00"
Xiaomi,"2,600,000.00"


In [ ]:
cmp=pd.DataFrame({'before':before,'after':after,'n':uc.groupby('Brand').size()})
cmp['change_%']= (cmp.after / cmp.before - 1) * 100
cmp[cmp.n >= 50].sort_values('change_%', key=abs, ascending=False).head(10)

,before,after,n,change_%
Brand,,,,
Honda,"750,000.00","710,000.00",238,-5.33
Fiat,"220,000.00","230,000.00",872,4.55
Porsche,"5,550,000.00","5,750,000.00",177,3.60
BMW,"1,800,000.00","1,750,000.00",1686,-2.78
Proton,"350,000.00","359,500.00",122,2.71
Audi,"2,200,000.00","2,150,000.00",363,-2.27
Volkswagen,"745,000.00","730,000.00",684,-2.01
Skoda,"1,000,000.00","980,000.00",824,-2.00
Nissan,"560,000.00","550,000.00",1133,-1.79


**the duplicates methodlgy: if the price doesn't shift by more than 10% ,then the drop is safe
why 10%? as we tried to drop the same persentage of rows to test if the shift in the median will be in the 10% range, and mostly it was in the range**

 **Why did the median price shift?**
When analyzing the effect of dropping duplicates, we observe that the median price for **Honda** shifted down by 5.33% (~40k). Below, we query Honda cars to analyze why the median shifted.

In [ ]:
uc[uc['Brand'] == 'Honda']
# ليه الـ median اتحرك 40 ألف؟
# لأن الصفوف اللي حوالين الـ median قليلة جدًا: 5 عربيات بس في ±5% من 750k. لما الوسط مش مزحوم، شيل 11 صف بيحرك الـ median بقفزة.

,Brand,Model,Trim,Year,Price,Mileage,Color,Fuel Type,Body Shape,Transmission,Engine,Engine Capacity,Cylinder Count,Turbo Count,HP,Drivetrain,car_age,km_per_year,Brand_Tier,Image Paths,Condition,Condition_Score,model_average_price,model_min_price,model_max_price,is_duplicated
7247,Honda,Accord,Base,2024,1930000,19000,Black,Gas,Sedan,Automatic,1.5L Turbo,1498,4,1,192,FWD,2,"9,500.00",3,Used_Images\Honda_Accord_2024\car_img_1.jpg; U...,Very Good,7.60,"1,554,843.75",215000,2200000,False
7248,Honda,Accord,Lx Turbo,2022,1700000,71000,Black,Gas,Sedan,Automatic,1.5L Turbo,1498,4,1,192,FWD,4,"17,750.00",3,Used_Images\Honda_Accord_2022\car_img_1.jpg; U...,Very Good,6.30,"1,554,843.75",215000,2200000,False
7249,Honda,Accord,Highline,2021,1350000,140000,Silver,Gas,Sedan,Automatic,1.5L Turbo,1498,4,1,192,FWD,5,"28,000.00",3,Used_Images\Honda_Accord_2021\car_img_1.jpg; U...,Good,5.90,"1,554,843.75",215000,2200000,False
7250,Honda,Accord,Base,1993,270000,418000,Grey,Gas,Sedan,Automatic,1.5L Turbo,1498,4,1,192,FWD,33,"12,666.70",3,Used_Images\Honda_Accord_1993\car_img_1.jpg; U...,Fair,2.70,"1,554,843.75",215000,2200000,False
7251,Honda,Accord,Highline,2021,1350000,140000,Silver,Gas,Sedan,Automatic,1.5L Turbo,1498,4,1,192,FWD,5,"28,000.00",3,Used_Images\Honda_Accord_2021\car_img_1.jpg; U...,Good,5.90,"1,554,843.75",215000,2200000,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7480,Honda,Cr-V,Base,2020,850000,115000,Black,Gas,SUV,Automatic,1.5L Turbo,1498,4,1,190,FWD,6,"19,166.70",3,Used_Images\honda_cr-v_2020_30067\img_1.jpg; U...,Good,5.80,"1,974,629.63",780000,3000000,False
7481,Honda,Hr-V,Base,2005,499000,330000,Blue,Gas,4X4,Manual,1.6L Vtec,1590,4,0,124,FWD,21,"15,714.30",3,Used_Images\honda_hr-v_2005_40801\img_1.jpeg; ...,Fair,3.50,"1,157,375.00",400000,2100000,False
7482,Honda,Hr-V,Base,2005,400000,350000,Blue,Gas,4X4,Automatic,1.6L Vtec,1590,4,0,124,FWD,21,"16,666.70",3,Used_Images\honda_hr-v_2005_55462\img_1.jpg; U...,Poor,1.20,"1,157,375.00",400000,2100000,False
7483,Honda,Hr-V,Vtec,2005,450000,210000,Blue,Gas,4X4,Automatic,1.6L Vtec,1590,4,0,124,FWD,21,"10,000.00",3,Used_Images\honda_hr-v_2005_73588\img_1.jpg; U...,Good,4.70,"1,157,375.00",400000,2100000,False


In [ ]:
uc=uc[~ uc.is_duplicated].drop(columns='is_duplicated').reset_index(drop=True)

In [ ]:
uc.shape

(23535, 25)

## Extreme values in 'price' & 'mileage'& 'km_per_year'

In [ ]:
uc[['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage','car_age','km_per_year']].describe(percentiles=[.99,.999,.01,.001])

,Year,Price,Mileage,car_age,km_per_year
count,"23,535.00","23,535.00","23,535.00","23,535.00","23,535.00"
mean,"2,015.28","1,166,918.18","132,937.52",10.72,"13,354.13"
std,7.60,"1,498,815.16","92,839.00",7.60,"6,829.44"
min,"1,964.00","50,000.00",0.00,0.00,0.00
0.1%,"1,976.53","54,068.00",0.00,0.00,0.00
1%,"1,992.00","95,000.00","1,000.00",1.00,10.00
50%,"2,017.00","700,000.00","120,000.00",9.00,"12,909.10"
99%,"2,025.00","7,500,000.00","400,000.00",34.00,"29,750.00"
99.9%,"2,026.00","14,946,600.00","519,320.00",49.47,"38,000.00"
max,"2,026.00","36,000,000.00","1,111,111.00",62.00,"50,000.00"


mileage

In [ ]:
uc.nsmallest(40,'Mileage')[['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage']]

,Brand,Model,Trim,Year,Price,Mileage
324,Audi,Q8,Base,2025,7300000,0
1618,BMW,X1,Base,2024,2975000,0
1623,BMW,X1,Base,2025,2950000,0
2199,BYD,F3,Base,2025,565000,0
2508,Chery,Arrizo 5,Base,2026,450000,0
2615,Chery,Tiggo 8,Base,2026,500000,0
5777,Fiat,Punto,Base,2026,155000,0
6734,Haval,H6,Base,2026,800000,0
9518,Jetour,X70S,Base,2026,1000000,0
10901,Kia,Sportage,Special Edition,2024,1990000,0


In [ ]:
uc[uc['Mileage']==0].value_counts().sum()

np.int64(37)

In [ ]:
uc.nlargest(30,'Mileage')[['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage','km_per_year']]

,Brand,Model,Trim,Year,Price,Mileage,km_per_year
12001,Mercedes-Benz,E 200,Base,1988,500000,1111111,"29,239.80"
5025,Daewoo,Racer,Base,1991,110000,1000000,"28,571.40"
5470,Fiat,Regata,Base,1986,70000,1000000,"25,000.00"
5480,Fiat,Regata,Base,1986,75000,1000000,"25,000.00"
5530,Fiat,127,Base,1976,50000,1000000,"20,000.00"
5551,Fiat,128,Base,1987,65000,1000000,"25,641.00"
5559,Fiat,128,Base,1989,55000,1000000,"27,027.00"
5618,Fiat,1500,Base,1968,210000,1000000,"17,241.40"
5677,Fiat,128,Base,1977,75000,1000000,"20,408.20"
5459,Fiat,Regata,Base,1984,58000,999900,"23,807.10"


In [ ]:
uc['is_zero_km'] = uc.Mileage == 0
uc['mileage_suspect'] = (uc.Mileage >= 999_000) | \
    uc.Mileage.astype(str).map(lambda s: len(s) >= 5 and len(set(s)) == 1)
print(uc.is_zero_km.sum(), uc.mileage_suspect.sum())

37 25


miles_per_year


In [ ]:
uc.nlargest(30,'km_per_year')[['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage','km_per_year']]

,Brand,Model,Trim,Year,Price,Mileage,km_per_year
2272,Changan,Cs55,Base,2025,1250000,50000,"50,000.00"
2301,Changan,Cs55,Flagship,2025,1250000,50000,"50,000.00"
2518,Chery,Arrizo 5,At Baseline Leather,2025,640000,50000,"50,000.00"
13928,MG,5,Luxury,2025,810000,50000,"50,000.00"
14032,MG,5,Base,2025,810000,50000,"50,000.00"
15765,Nissan,Qashqai,Base,2025,1320000,50000,"50,000.00"
2430,Chery,Arrizo 5,Base,2025,575000,47000,"47,000.00"
2460,Chery,Tiggo 4,Pro,2025,820000,45000,"45,000.00"
7099,Hyundai,Elantra,Cn7 Redline,2025,1400000,45000,"45,000.00"
15481,Nissan,Sentra,Base,2025,760000,45000,"45,000.00"


In [ ]:
uc.nsmallest(30,'km_per_year')[['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage','km_per_year']]

,Brand,Model,Trim,Year,Price,Mileage,km_per_year
197,Audi,A3,Base,2026,2390000,10,0.00
198,Audi,A3,Base,2026,2390000,10,0.00
215,Audi,Q2,Base,2026,2150000,10,0.00
233,Audi,Q3,Base,2026,2890000,10,0.00
234,Audi,Q3,Base,2026,2890000,10,0.00
324,Audi,Q8,Base,2025,7300000,0,0.00
423,BMW,320I,Sport,2026,3150000,2500,0.00
426,BMW,320I,Sport,2026,3150000,1000,0.00
427,BMW,320I,Sport,2026,3175000,4000,0.00
430,BMW,320I,Luxury-Line,2026,2875000,5000,0.00


In [ ]:
# 0 km/year an 2026 model=> nan
mask = (uc.car_age == 0) & (uc.km_per_year == 0)
uc.loc[mask, 'km_per_year'] = np.nan

price

In [ ]:
uc.nsmallest(30,'Price')[['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage']]

,Brand,Model,Trim,Year,Price,Mileage
5410,Fiat,Polonez,Base,1988,50000,200000
5528,Fiat,126,Base,1991,50000,130000
5530,Fiat,127,Base,1976,50000,1000000
5531,Fiat,127,Base,1982,50000,100000
5534,Fiat,127,Base,1987,50000,300000
5548,Fiat,128,Base,1981,50000,111111
5550,Fiat,128,Base,1987,50000,100000
5555,Fiat,128,Base,1985,50000,420500
5560,Fiat,128,Base,1990,50000,200000
5568,Fiat,128,Base,1988,50000,100000


In [ ]:
uc.nlargest(30,'Price')[['Brand', 'Model', 'Trim', 'Year', 'Price', 'Mileage','car_age','km_per_year']]

,Brand,Model,Trim,Year,Price,Mileage,car_age,km_per_year
5260,Ferrari,Purosangue,Base,2025,36000000,3400,1,"3,400.00"
19901,Rolls-Royce,Cullinan,Base,2022,28900000,11000,4,"2,750.00"
400,Bentley,Continental Gt,Base,2023,25000000,37000,3,"12,333.30"
11386,Lamborghini,Urus,Base,2025,24600000,7000,1,"7,000.00"
11387,Lamborghini,Urus,Base,2025,24600000,7000,1,"7,000.00"
407,Bentley,Bentayga,Base,2023,23000000,7000,3,"2,333.30"
40,Aston Martin,Db,Base,2025,22500000,2401,1,"2,401.00"
19902,Rolls-Royce,Spectre,Base,2023,21500000,16100,3,"5,366.70"
11388,Lamborghini,Urus,Base,2021,20000000,15000,5,"3,000.00"
399,Bentley,Continental Gt,Base,2020,19047619,22000,6,"3,666.70"


## Engine issues cleaning

In [ ]:
sus=uc[(uc.Engine=='Electric')&(uc['Fuel Type'] != 'Electric')]
sus[['Brand','Cylinder Count','Fuel Type','Engine','Engine Capacity']].head(20)

,Brand,Cylinder Count,Fuel Type,Engine,Engine Capacity
365,Avatr,0,Gas,Electric,0
368,Avatr,0,Gas,Electric,0
372,Avatr,0,Gas,Electric,0
442,BMW,0,Gas,Electric,0
1217,BMW,0,Gas,Electric,0
1608,BMW,0,Gas,Electric,0
1824,BMW,0,Gas,Electric,0
1858,BMW,0,Gas,Electric,0
1870,BMW,0,Gas,Electric,0
1871,BMW,0,Gas,Electric,0


In [ ]:
sus['Brand'].value_counts().sum()

np.int64(254)

In [ ]:
sus.groupby(['Brand','Model']).size().sort_values(ascending=False).head(15)

Brand          Model  
Hyundai        Matrix     124
               Ix35        56
Geely          Gx3 Pro     25
Hyundai        I30         15
BMW            Ix           5
Hyundai        I10          3
               Getz         3
Avatr          12           2
BMW            I3           2
Mercedes-Benz  Eqa 260      2
Dodge          Durango      2
Volvo          Xc30         2
Deepal         S07          1
Avatr          7            1
Daihatsu       Pickup       1
dtype: int64

In [ ]:
cols = ['Engine Capacity', 'Cylinder Count']
uc[cols] = uc[cols].astype('Int64')

# ---------- A: electric vs combustion ----------
ev_models = [('BMW','Ix'),('BMW','I3'),('Avatr','12'),('Avatr','7'),('Mercedes-Benz','Eqa 260'),
             ('BYD','Seagull'),('BYD','Atto 3'),('Im Motors','Ls7'),('Lotus','Eletre'),
             ('Nissan','Leaf'),('Volkswagen','Id3'),('Dongfeng','Box')]
is_ev_model = uc[['Brand','Model']].apply(tuple, axis=1).isin(ev_models)

uc.loc[(uc.Engine == 'Electric') & is_ev_model, 'Fuel Type'] = 'Electric'
ev = uc['Fuel Type'] == 'Electric'
uc.loc[ev, cols] = pd.NA
uc.loc[ev, 'Engine'] = 'Electric'
uc.loc[ev, 'Turbo Count'] = 0
fake = (uc.Engine == 'Electric') & ~ev
uc.loc[fake, ['Engine'] + cols] = pd.NA
print("Electric count:", ev.sum(), "Fake Electric count:", fake.sum())

# ---------- B: flag conflicts ----------
uc['fuel_engine_conflict'] = (
    (uc.Engine.str.contains('Diesel', na=False) & (uc['Fuel Type'] != 'Diesel')) |
    (uc.Engine.str.contains('Hybrid|Plug', case=False, na=False) & (uc['Fuel Type'] != 'Hybrid')))
print("Total conflicts flagged:", uc.fuel_engine_conflict.sum())

# ---------- C: impossible specs ----------
m3 = (uc.Brand == 'BMW') & (uc.Model == 'M3') & (uc['Cylinder Count'] == 4)
uc.loc[m3, 'Cylinder Count'] = 6
bad = uc.Engine.isin(['11.1L Diesel', '4.6L Diesel'])
uc.loc[bad, ['Engine'] + cols] = pd.NA
print("Fixed M3 rows:", m3.sum(), "Cleared bad specs rows:", bad.sum())
print("Total missing values per column after cleaning:\n", uc[cols].isna().sum())


Electric count: 213 Fake Electric count: 235
Total conflicts flagged: 68
Fixed M3 rows: 1 Cleared bad specs rows: 19
Total missing values per column after cleaning:
 Engine Capacity    467
Cylinder Count     467
dtype: int64


## The Engine Imputation Pipeline (The "Donor" Logic)

In [ ]:
target=(uc['Fuel Type']!='Electric')& uc['Engine'].isna()
target.sum()

np.int64(254)

In [ ]:
healthy = (~target) & (uc['Fuel Type'] != 'Electric') & uc['Engine'].notna() \
          & (uc['Engine Capacity'] > 0).fillna(False)

spec = (uc['Engine'].astype(str) + '|' + uc['Engine Capacity'].astype(str)
        + '|' + uc['Cylinder Count'].astype(str))

ref = (uc[healthy].assign(spec=spec[healthy])
         .groupby(['Brand', 'Model'], observed=True)['spec']
         .agg(n_donors='size', n_specs='nunique', spec='first')
         .reset_index())
print(ref.shape, healthy.sum())

(839, 5) 23068


In [ ]:
tg = (uc.loc[target, ['Brand', 'Model']].reset_index(names='row')
        .merge(ref, how='left', on=['Brand', 'Model']).set_index('row'))
ok = (tg.n_specs == 1) & (tg.n_donors >= 3)
print(ok.sum(), (~ok).sum())

163 91


In [ ]:
parts = tg.loc[ok, 'spec'].str.split('|', expand=True)
idx = parts.index

uc['engine_imputed'] = False
uc.loc[idx, 'Engine'] = parts[0]
uc.loc[idx, 'Engine Capacity'] = pd.to_numeric(parts[1]).astype('Int64')
uc.loc[idx, 'Cylinder Count'] = pd.to_numeric(parts[2]).astype('Int64')
uc.loc[idx, 'engine_imputed'] = True

In [ ]:
print(uc.engine_imputed.sum(), uc['Engine'].isna().sum(),
      uc['Engine Capacity'].isna().sum(), uc['Cylinder Count'].isna().sum())


163 91 304 304


## Data types handling

In [ ]:
cat_cols = ['Brand','Model','Trim','Color','Fuel Type','Body Shape',
            'Transmission','Drivetrain','Engine']
uc[cat_cols] = uc[cat_cols].astype('category')

uc['Condition'] = pd.Categorical(uc['Condition'],
    categories=['Poor','Fair','Good','Very Good','Excellent'], ordered=True)

print(uc.dtypes)
print(round(uc.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Brand                   category
Model                   category
Trim                    category
Year                       int64
Price                      int64
Mileage                    int64
Color                   category
Fuel Type               category
Body Shape              category
Transmission            category
Engine                  category
Engine Capacity            Int64
Cylinder Count             Int64
Turbo Count                int64
HP                         int64
Drivetrain              category
car_age                    int64
km_per_year              float64
Brand_Tier                 int64
Image Paths               object
Condition               category
Condition_Score          float64
model_average_price      float64
model_min_price            int64
model_max_price            int64
is_zero_km                  bool
mileage_suspect             bool
fuel_engine_conflict        bool
engine_imputed              bool
dtype: object
13.5 MB


# price history Cleaning

## EDA

In [ ]:
ph.shape

(67271, 7)

In [ ]:
ph.duplicated().sum()

np.int64(189)

In [ ]:
ph.columns

Index(['Brand', 'Model', 'Year', 'Month/Year', 'Average Price', 'Min Price',
       'Max Price'],
      dtype='object')

In [ ]:
ph.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67271 entries, 0 to 67270
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Brand          67271 non-null  object
 1   Model          67271 non-null  object
 2   Year           67271 non-null  int64 
 3   Month/Year     67271 non-null  object
 4   Average Price  67271 non-null  int64 
 5   Min Price      67271 non-null  int64 
 6   Max Price      67271 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 3.6+ MB


In [ ]:
ph['Year'].value_counts(ascending=False)

,count
Year,
2015,4240
2016,4115
2008,3824
2010,3775
2011,3563
2007,3421
2014,3417
2017,3361
2013,3357


## Duplicates

In [ ]:
ph=ph.drop_duplicates().reset_index(drop=True)

## update data types

In [ ]:
ph['date'] = pd.to_datetime(ph['Month/Year'], format='%b-%y')

## inappropriate row has been removed

In [ ]:
bad = (ph['Min Price'] > ph['Average Price']) | (ph['Average Price'] > ph['Max Price'])
ph[bad]

,Brand,Model,Year,Month/Year,Average Price,Min Price,Max Price,date
14955,Fiat,Siena,2002,Jul-14,38000,37500,37500,2014-07-01


In [ ]:
ph = ph[~bad].reset_index(drop=True)
# "تم حذف صف شاذ بدل تعديله"

In [ ]:
bad.sum()
ph.shape

(67081, 8)

## 'Brand' Cleaning

In [ ]:
land = ph.Brand == 'Land'
ph.loc[land, 'Model'] = (ph.loc[land, 'Model']
                         .str.replace('Rover Range Rover ', '', regex=False)
                         .str.replace('Rover ', '', regex=False))
ph.loc[land, 'Brand'] = 'Range Rover'

ssang = ph.Brand == 'Ssang'
ph.loc[ssang, 'Model'] = ph.loc[ssang, 'Model'].str.replace('Yong ', '', regex=False)
ph.loc[ssang, 'Brand'] = 'SsangYong'

ph.loc[ph.Brand == 'Emgrand', 'Brand'] = 'Geely'
ph['Brand'] = ph['Brand'].replace({'Mercedes': 'Mercedes-Benz', 'Byd': 'BYD', 'Jac': 'JAC'})

print(ph.Brand.nunique())
print(set(ph.Brand) - set(uc.Brand))

43
set()


In [ ]:
ph['Brand'].nunique()


43

## drop jan-2026 and dec-2025 as they include less serieses which make the shift measuring on them unaccuarate

In [ ]:
print(ph.date.max())                                   # 2026-01-01
ph = ph[ph.date <= '2025-11-01'].reset_index(drop=True)
print(ph.date.max(), ph.shape)                         # 2025-11-01, about (66690, 8)

2026-01-01 00:00:00
2025-11-01 00:00:00 (66689, 8)


## keeping only data fron jan-2016


In [ ]:
start = '2016-01-01'
ph=ph[ph.date >= start]

In [ ]:


def norm(x):
    return re.sub(r'[\s\-_]+', '', str(x).lower())

canon = {}
for b, m in ph[['Brand', 'Model']].drop_duplicates().itertuples(index=False):
    canon.setdefault((norm(b), norm(m)), set()).add(m)
canon = {k: next(iter(v)) for k, v in canon.items() if len(v) == 1}

old = uc['Model'].astype(str)
new = [canon.get((norm(b), norm(m)), m) for b, m in zip(uc['Brand'], uc['Model'])]
print((pd.Series(new, index=uc.index) != old).sum())
uc['Model'] = new

2682


# Data downloding

In [ ]:
ph.to_csv('/content/drive/MyDrive/DEPI datasets/price_history(super cleaned)1.1.csv', index=False)
